# TrendPulse - Part 2: NumPy and Pandas

## 1. NumPy arrays

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

COUNTRIES = ["US", "GB", "IN"]

frames = []
for country in COUNTRIES:
    df_country = pd.read_csv(f"/content/data/{country}videos.csv")
    df_country["country"] = country
    frames.append(df_country)

df = pd.concat(frames, ignore_index=True)
print(df.shape)

(46230, 17)


In [3]:
views = df["views"].fillna(0).to_numpy()
likes = df["likes"].fillna(0).astype("int64").to_numpy()
dislikes = df["dislikes"].fillna(0).astype("int64").to_numpy()
comments = df["comment_count"].fillna(0).astype("int64").to_numpy()

In [4]:
arrays = {"views": views, "likes": likes, "dislikes": dislikes, "comments": comments}

for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

views: shape=(46230,), dtype=int64
likes: shape=(46230,), dtype=int64
dislikes: shape=(46230,), dtype=int64
comments: shape=(46230,), dtype=int64


In [5]:
# 10 largest view counts
top10 = np.sort(views)[::-1][:10]
print("Top 10 views:", top10)

# every 100th record
every_100th = views[::100]
print("Every 100th record:", every_100th[:10], "... total", len(every_100th))

# a sub-array: first 5 rows of views and likes
stats = np.column_stack((views, likes, dislikes, comments))
sub = stats[:5, :2]
print(sub)

Top 10 views: [360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]
Every 100th record: [ 48555  21869 245258 421352  54521 114671 115468 315213  98662 219533] ... total 463
[[ 48555   2189]
 [198385   1159]
 [231656   8742]
 [  8083    176]
 [373161  11305]]


## 2. Engagement score

In [6]:
total_ratings = likes + dislikes

like_ratio = np.divide(likes, total_ratings,
                       out=np.zeros(len(likes)), where=total_ratings != 0)

comment_rate = np.divide(comments, views,
                         out=np.zeros(len(views)), where=views != 0)

print(like_ratio[:5])
print(comment_rate[:5])

[0.92401857 0.70199879 0.90703465 0.95652174 0.98544282]
[0.00280095 0.00728886 0.00186483 0.00235061 0.00238771]


In [7]:
def min_max(arr):
    low = arr.min()
    high = arr.max()
    if high == low:
        return np.zeros(len(arr))
    return (arr - low) / (high - low)

views_norm = min_max(views)
likes_norm = min_max(likes)
comments_norm = min_max(comments)

print(views_norm.min(), views_norm.max())

0.0 1.0


In [8]:
W_VIEWS = 0.2
W_LIKES = 0.4
W_COMMENTS = 0.4

engagement_score = (W_VIEWS * views_norm
                    + W_LIKES * likes_norm
                    + W_COMMENTS * comments_norm)

df["engagement_score"] = engagement_score

print(engagement_score[:5])
print("Max score:", engagement_score.max())

[1.28011379e-04 8.69298271e-04 4.76814405e-04 1.62103576e-05
 8.28363153e-04]
Max score: 0.907609296111204


In [9]:
log_views = np.log1p(views)

print("Raw views: min", views.min(), "max", views.max())
print("Log views: min", log_views.min().round(2), "max", log_views.max().round(2))

Raw views: min 1066 max 360760164
Log views: min 6.97 max 19.7


In [10]:
threshold = np.percentile(engagement_score, 90)
top_mask = engagement_score >= threshold

print("Threshold score:", threshold)
print("Records in top 10%:", top_mask.sum(), "of", len(top_mask))
print("Average views, top 10%:", views[top_mask].mean().round(0))
print("Average views, others:", views[~top_mask].mean().round(0))

Threshold score: 0.005672828324812643
Records in top 10%: 4623 of 46230
Average views, top 10%: 7707941.0
Average views, others: 453121.0


## 3. Reshape and broadcasting

In [11]:
df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m")

daily = pd.crosstab(df["trending_date"], df["country"])
country_order = daily.columns.tolist()
counts = daily.to_numpy()

print("Country order:", country_order)
print("Shape (days, countries):", counts.shape)
print(counts[:3])

Country order: ['GB', 'IN', 'US']
Shape (days, countries): (80, 3)
[[200 200 200]
 [200 201 202]
 [200 200 200]]


In [12]:
flat = counts.flatten()
print("Flattened shape:", flat.shape)

back = flat.reshape(counts.shape)
print("Reshaped back:", back.shape)
print("Same as the original:", np.array_equal(back, counts))

by_country = counts.reshape(len(country_order), -1) if counts.size % len(country_order) == 0 else None
print("Another reshape:", None if by_country is None else by_country.shape)

Flattened shape: (240,)
Reshaped back: (80, 3)
Same as the original: True
Another reshape: (3, 80)


In [13]:
day_totals = counts.sum(axis=1, keepdims=True)
share = counts / day_totals

print("day_totals shape:", day_totals.shape)
print(share[:3].round(3))
print("Each row adds up to 1:", share.sum(axis=1)[:5])

day_totals shape: (80, 1)
[[0.333 0.333 0.333]
 [0.332 0.333 0.335]
 [0.333 0.333 0.333]]
Each row adds up to 1: [1. 1. 1. 1. 1.]


In [14]:
country_avg = counts.mean(axis=0)
relative = counts / country_avg

print("country_avg:", country_avg.round(1))
print(relative[:3].round(2))
print("Average of each column is 1:", relative.mean(axis=0).round(2))

country_avg: [188.4 188.4 201. ]
[[1.06 1.06 1.  ]
 [1.06 1.07 1.  ]
 [1.06 1.06 1.  ]]
Average of each column is 1: [1. 1. 1.]


## 4. Loading and inspection

In [15]:
dfs = dict(zip(COUNTRIES, frames))

for country, d in dfs.items():
    print("=" * 50)
    print(f"{country}  shape: {d.shape}")
    display(d.head(3))
    d.info()
    display(d.describe())

US  shape: (16080, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_disabled        16080 non-null  bool   
 14  video_error_or_removed  16080 non-null

,category_id,views,likes,dislikes,comment_count
count,16080.000000,1.608000e+04,1.607600e+04,16080.000000,16068.000000
mean,18.975871,1.012289e+06,3.950464e+04,2115.772015,3118.737989
std,7.590729,3.746761e+06,1.719414e+05,7973.927750,10060.176081
min,1.000000,1.621000e+03,0.000000e+00,0.000000,0.000000
25%,10.000000,1.173078e+05,3.353000e+03,194.000000,375.000000
50%,23.000000,2.890725e+05,9.229000e+03,540.000000,999.000000
75%,24.000000,7.507190e+05,2.626700e+04,1506.000000,2613.250000
max,28.000000,1.292292e+08,6.795138e+06,247546.000000,318346.000000


GB  shape: (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,17.14.11,Wild Winter Prank,OfficialRecipe78,28,2017-10-18T07:42:02.000Z,[none],84344,2155.0,163,475.0,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14T12:56:18.000Z,Minecraft|Paris,230434,5412.0,330,1704.0,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,UltimateReview54,23,2017-11-13T06:19:10.000Z,Paris,338852,16813.0,1333,1385.0,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB


... (2 outputs left out)

IN  shape: (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                15075 non-null  object 
 1   trending_date           15075 non-null  object 
 2   title                   15075 non-null  object 
 3   channel_title           15075 non-null  object 
 4   category_id             15075 non-null  int64  
 5   publish_time            15075 non-null  object 
 6   tags                    15075 non-null  object 
 7   views                   15075 non-null  int64  
 8   likes                   15071 non-null  float64
 9   dislikes                15075 non-null  int64  
 10  comment_count           15063 non-null  float64
 11  thumbnail_link          15075 non-null  object 
 12  comments_disabled       15075 non-null  bool   
 13  ratings_disabled        15075 non-null  bool   
 14  video_error_or_removed  15075 non-null

,category_id,views,likes,dislikes,comment_count
count,15075.000000,1.507500e+04,1.507100e+04,15075.000000,15063.000000
mean,19.836086,9.545924e+05,3.321833e+04,1937.757877,3210.599283
std,7.341271,3.035647e+06,1.042103e+05,5791.482855,8251.638153
min,1.000000,3.542000e+03,0.000000e+00,0.000000,0.000000
25%,17.000000,1.521655e+05,4.107000e+03,257.000000,496.000000
50%,24.000000,3.558020e+05,1.091500e+04,681.000000,1243.000000
75%,24.000000,8.639980e+05,2.926200e+04,1815.500000,3101.000000
max,28.000000,1.232081e+08,3.634665e+06,287063.000000,346103.000000


In [16]:
print("Combined shape:", df.shape)
display(df.head(3))
df.info()
display(df.describe())

Combined shape: (46230, 18)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,engagement_score
0,g_N-w8LeAi8,2017-11-14,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.000128
1,Ugyql2NlPAd,2017-11-14,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.000869
2,8fE19jkxJ9n,2017-11-14,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US,0.000477


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   video_id                46230 non-null  object        
 1   trending_date           46230 non-null  datetime64[ns]
 2   title                   46230 non-null  object        
 3   channel_title           46230 non-null  object        
 4   category_id             46230 non-null  int64         
 5   publish_time            46230 non-null  object        
 6   tags                    46230 non-null  object        
 7   views                   46230 non-null  int64         
 8   likes                   46218 non-null  float64       
 9   dislikes                46230 non-null  int64         
 10  comment_count           46194 non-null  float64       
 11  thumbnail_link          46230 non-null  object        
 12  comments_disabled       46230 non-null  bool  

,trending_date,category_id,views,likes,dislikes,comment_count,engagement_score
count,46230,46230.000000,4.623000e+04,4.621800e+04,46230.000000,46194.000000,4.623000e+04
mean,2017-12-21 21:18:10.979883264,18.645339,1.178603e+06,4.580005e+04,2325.983625,3449.773347,3.089299e-03
min,2017-11-14 00:00:00,1.000000,1.066000e+03,0.000000e+00,0.000000,0.000000,9.205946e-07
25%,2017-12-03 00:00:00,10.000000,1.294632e+05,3.656000e+03,213.000000,404.000000,3.670677e-04
50%,2017-12-22 00:00:00,22.000000,3.182775e+05,1.020100e+04,594.000000,1074.000000,9.283343e-04
75%,2018-01-10 00:00:00,24.000000,8.240505e+05,2.898775e+04,1666.000000,2779.000000,2.352449e-03
max,2018-02-01 00:00:00,28.000000,3.607602e+08,2.753782e+07,618175.000000,778390.000000,9.076093e-01
std,NaN,7.646745,5.916188e+06,2.969554e+05,10518.962756,13665.153495,1.400142e-02


In [17]:
# trending_date was already converted in Task 3
df["publish_time"] = pd.to_datetime(df["publish_time"], utc=True).dt.tz_localize(None)

df["category_id"] = df["category_id"].astype(int)
df["views"] = df["views"].astype("int64")

for col in ["comments_disabled", "ratings_disabled", "video_error_or_removed"]:
    df[col] = df[col].astype(bool)

print(df[["trending_date", "publish_time", "category_id", "views",
          "comments_disabled", "ratings_disabled", "video_error_or_removed"]].dtypes)

trending_date             datetime64[ns]
publish_time              datetime64[ns]
category_id                        int64
views                              int64
comments_disabled                   bool
ratings_disabled                    bool
video_error_or_removed              bool
dtype: object


In [18]:
print("--- Missing values per column ---")
print(df.isna().sum())

dup = df.duplicated(subset=["video_id", "trending_date", "country"]).sum()
print("\nDuplicate rows (same video, same day, same country):", dup)

print("\nRemoved or errored videos:", df["video_error_or_removed"].sum())
print("Comments disabled:", df["comments_disabled"].sum())
print("Ratings disabled:", df["ratings_disabled"].sum())
print("Records with 0 views:", (df["views"] == 0).sum())

published_after = (df["publish_time"].dt.normalize() > df["trending_date"]).sum()
print("Published after the trending date:", published_after)

print("\nTrending dates:", df["trending_date"].min().date(), "to", df["trending_date"].max().date())
print("Publish dates: ", df["publish_time"].min().date(), "to", df["publish_time"].max().date())

--- Missing values per column ---
video_id                     0
trending_date                0
title                        0
channel_title                0
category_id                  0
publish_time                 0
tags                         0
views                        0
likes                       12
dislikes                     0
comment_count               36
thumbnail_link               0
comments_disabled            0
ratings_disabled             0
video_error_or_removed       0
description               1624
country                      0
engagement_score             0
dtype: int64

Duplicate rows (same video, same day, same country): 230

Removed or errored videos: 92
Comments disabled: 1048
Ratings disabled: 544
Records with 0 views: 0
Published after the trending date: 0

Trending dates: 2017-11-14 to 2018-02-01
Publish dates:  2017-10-09 to 2018-02-01


## 5. Filtering

In [19]:
cols = ["title", "channel_title", "country", "views", "likes", "engagement_score"]

mask_a = (df["views"] >= 1_000_000) & (df["likes"] > 50_000)
result_a = df.loc[mask_a, cols]

print("Records:", len(result_a))
display(result_a.head(10))

Records: 6757


,title,channel_title,country,views,likes,engagement_score
7,Official Cats Speech EXCLUSIVE,AmazingSecret45,US,2841278,52635.0,0.008540
11,Surprising Budget Interview WATCH!,Brand NewReview23,US,2855690,160954.0,0.008996
13,DJ Aster - Festival (Live),WildUnboxing39,US,1905264,75034.0,0.004453
19,Shocking Monsoon Reaction,UltimateReview24,US,1130682,78147.0,0.003808
55,Epic Election Interview,UltimateUnboxing65,US,2998582,120472.0,0.009890
69,Mira Solis - Guitar (Official Video) FULL,FinalTrailer17,US,1624900,69974.0,0.003877
122,Nova Kane - Pasta (Lyric Video),ShockingSecret98,US,3501006,123821.0,0.003739
141,DJ Aster - Football (Official Video),FinalTrailer17,US,1082103,58664.0,0.002720
159,Blue Harbor - Cars (Live),QuietReview27,US,3536892,128388.0,0.007517
163,Crazy Minecraft Prank,IncredibleTrailer23,US,1250367,61091.0,0.003056


In [20]:
top20 = df.sort_values("engagement_score", ascending=False).head(20)
display(top20[cols])

,title,channel_title,country,views,likes,engagement_score
26526,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822.0,0.907609
26325,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701.0,0.721323
24108,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793.0,0.707360
30954,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910.0,0.670311
26123,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,236275528,18035565.0,0.594427
23907,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,200444300,8959133.0,0.555234
30753,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691.0,0.552210
25922,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,210747294,16086924.0,0.530203
30552,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,230704137,6439282.0,0.450466
23707,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,159635918,7135146.0,0.442194


In [21]:
categories = pd.read_csv("/content/data/categories.csv")
print(categories.columns.tolist())

music_id = int(categories.loc[categories["category_name"] == "Music", "category_id"].iloc[0])
gaming_id = int(categories.loc[categories["category_name"] == "Gaming", "category_id"].iloc[0])
print("Music id:", music_id, " Gaming id:", gaming_id)

median_views = df["views"].median()
print("Median views:", median_views)

in_category = (df["category_id"] == music_id) | (df["category_id"] == gaming_id)
mask_c = in_category & (df["views"] > median_views)

result_c = df.loc[mask_c, cols]
print("Records:", len(result_c))
display(result_c.head(10))

['category_id', 'category_name']
Music id: 10  Gaming id: 20
Median views: 318277.5
Records: 9181


,title,channel_title,country,views,likes,engagement_score
13,DJ Aster - Festival (Live),WildUnboxing39,US,1905264,75034.0,0.004453
17,Brand New Guitar Highlights?,CrazyHighlights46,US,481623,21344.0,0.001754
20,The Hollow Pines - Guitar (Official Video),Brand NewSecret47,US,959060,38244.0,0.001920
21,Rook & Vale - Pasta (Official Video),PerfectTrailer65,US,388402,13318.0,0.000650
23,Lena Ortiz - Cricket (Official Video),FinalTrailer17,US,622912,26525.0,0.001364
49,DJ Aster - Guitar (Lyric Video) WATCH,HonestUnboxing68,US,766994,36646.0,0.001636
63,The Hollow Pines - Paris (Live),QuietInterview9,US,767987,35431.0,0.002021
69,Mira Solis - Guitar (Official Video) FULL,FinalTrailer17,US,1624900,69974.0,0.003877
82,Sunny Park - Cricket (Lyric Video) FINALE,QuietChallenge75,US,412775,14099.0,0.000788
87,ROOK & VALE - MINECRAFT (LYRIC VIDEO)!!,FinalTrailer17,US,924126,22901.0,0.001555


In [22]:
display(df.iloc[:5, :4])
display(df.loc[mask_a, cols].sort_values("views", ascending=False).head(5))

,video_id,trending_date,title,channel_title
0,g_N-w8LeAi8,2017-11-14,Honest Football Prank LEAKED,QuietSpeech10
1,Ugyql2NlPAd,2017-11-14,Surprising Cricket Review,IncredibleChallenge41
2,8fE19jkxJ9n,2017-11-14,Mira Solis - Football (Live) OFFICIAL,WildTutorial42
3,WOY0nn7HCsO,2017-11-14,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30
4,2Lc_Cs5VVr_,2017-11-14,Surprising Election Recipe,UltimateVlog96


,title,channel_title,country,views,likes,engagement_score
26526,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822.0,0.907609
30954,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910.0,0.670311
26325,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701.0,0.721323
30753,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691.0,0.552210
24108,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793.0,0.707360


## 6. Missing values and duplicates

In [23]:
missing = df.isna().sum()
missing = missing[missing > 0]

report = pd.DataFrame({
    "missing_count": missing,
    "percent": (missing / len(df) * 100).round(2)
})
display(report)

,missing_count,percent
likes,12,0.03
comment_count,36,0.08
description,1624,3.51


In [24]:
mask = df["likes"].isna() | df["comment_count"].isna()
display(df.loc[mask, ["title", "views", "likes", "comment_count",
                      "comments_disabled", "ratings_disabled"]].head(10))

,title,views,likes,comment_count,comments_disabled,ratings_disabled
1583,Amazing Budget Recipe FINALE?!,93299,NaN,378.0,False,False
3369,Perfect Football Reaction,339311,4020.0,NaN,False,False
3671,Brand New Robots Secret,3259919,103996.0,NaN,False,False
4698,Nova Kane - Winter (Official Video),1792339,79435.0,NaN,False,False
4830,Mira Solis - Football (Official Video)!!,992267,NaN,2569.0,False,False
5917,Incredible Phones Unboxing!,184440,7587.0,NaN,False,False
5923,DJ Aster - Bangalore (Live) FINALE,1092168,55664.0,NaN,False,False
7306,Insane Space Secret WATCH,77004,NaN,167.0,False,False
8738,Perfect Space Finale,211245,5964.0,NaN,False,False
9065,INSANE SPACE CHALLENGE,1754841,60632.0,NaN,False,False


In [25]:
df["likes"] = df["likes"].fillna(0).astype("int64")
df["comment_count"] = df["comment_count"].fillna(0).astype("int64")
df["description"] = df["description"].fillna("No description")

print("Missing values left:", df.isna().sum().sum())

Missing values left: 0


In [26]:
before = len(df)
df = df.dropna(subset=["video_id", "trending_date", "title", "channel_title"])
print("Rows dropped:", before - len(df))

Rows dropped: 0


In [27]:
before = len(df)
df = df.drop_duplicates(subset=["video_id", "trending_date", "country"], keep="first")
df = df.reset_index(drop=True)

print("Duplicates removed:", before - len(df))
print("Rows left:", len(df))

Duplicates removed: 230
Rows left: 46000


## 7. Joining category names

In [28]:
categories = pd.read_csv("/content/drive/MyDrive/data/categories.csv")
categories = categories[["category_id", "category_name"]]

print(categories.shape)
print("Each category_id appears once:", categories["category_id"].is_unique)
display(categories.head())

(17, 2)
Each category_id appears once: True


,category_id,category_name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


In [29]:
rows_before = len(df)

df = df.merge(categories, on="category_id", how="left")

print("Rows before:", rows_before)
print("Rows after: ", len(df))
print("Records without a category name:", df["category_name"].isna().sum())
display(df[["title", "category_id", "category_name"]].head())

Rows before: 46000
Rows after:  46000
Records without a category name: 0


,title,category_id,category_name
0,Honest Football Prank LEAKED,22,People & Blogs
1,Surprising Cricket Review,25,News & Politics
2,Mira Solis - Football (Live) OFFICIAL,10,Music
3,Zed Rivera - Space (Lyric Video)?!,10,Music
4,Surprising Election Recipe,27,Education


In [30]:
print(df["category_name"].value_counts())

category_name
Entertainment           12270
Music                   10310
People & Blogs           3804
Comedy                   3258
News & Politics          2422
Film & Animation         2297
Gaming                   2057
Sports                   1921
Howto & Style            1825
Education                1645
Science & Technology     1562
Autos & Vehicles         1007
Pets & Animals            913
Travel & Events           709
Name: count, dtype: int64


## 8. Aggregation

In [31]:
by_category = df.groupby("category_name").agg(
    records=("video_id", "count"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean")
).round(3).sort_values("avg_views", ascending=False)

display(by_category)

,records,avg_views,avg_likes,avg_engagement
category_name,,,,
Music,10310,3190950.477,134940.252,0.007
Entertainment,12270,901025.649,28932.230,0.003
Comedy,3258,603475.665,28692.015,0.002
Film & Animation,2297,577974.024,17191.444,0.002
People & Blogs,3804,551234.121,19835.409,0.002
Gaming,2057,509664.939,20747.269,0.002
Sports,1921,374978.164,10451.313,0.001
Education,1645,362716.266,12615.622,0.001
Science & Technology,1562,356350.413,11354.096,0.001


In [32]:
by_country = df.groupby("country").agg(
    records=("video_id", "count"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean")
).round(3)

display(by_country)

,records,avg_views,avg_likes,avg_engagement
country,,,,
GB,15000,1584041.317,65249.492,0.004
IN,15000,952100.764,33135.467,0.003
US,16000,1013904.378,39550.402,0.003


In [33]:
counts = df["category_name"].value_counts()
percent = (df["category_name"].value_counts(normalize=True) * 100).round(2)

distribution = pd.DataFrame({"records": counts, "percent": percent})
display(distribution)

,records,percent
category_name,,
Entertainment,12270,26.67
Music,10310,22.41
People & Blogs,3804,8.27
Comedy,3258,7.08
News & Politics,2422,5.27
Film & Animation,2297,4.99
Gaming,2057,4.47
Sports,1921,4.18
Howto & Style,1825,3.97


In [34]:
videos = df.sort_values("views").drop_duplicates(["video_id", "country"], keep="last").copy()
print("Unique videos:", len(videos))

videos["publish_weekday"] = videos["publish_time"].dt.day_name()
videos["publish_hour"] = videos["publish_time"].dt.hour

week_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

by_weekday = videos.groupby("publish_weekday")["views"].agg(["count", "mean", "median"]).reindex(week_order).round(0)
display(by_weekday)

by_hour = videos.groupby("publish_hour")["views"].agg(["count", "mean", "median"]).round(0)
display(by_hour.sort_values("mean", ascending=False).head(5))

Unique videos: 15059


,count,mean,median
publish_weekday,,,
Monday,2151,1192644.0,272614.0
Tuesday,2291,1009513.0,267805.0
Wednesday,2168,932667.0,272336.0
Thursday,2125,928150.0,241846.0
Friday,2075,1020493.0,258678.0
Saturday,2159,850419.0,266188.0
Sunday,2090,786213.0,256792.0


,count,mean,median
publish_hour,,,
12,607,2104435.0,264964.0
8,624,1393548.0,257374.0
16,637,1197658.0,282269.0
14,619,1104800.0,239087.0
21,651,1089720.0,273063.0


## 9. Time to trend and trending lifespan

In [35]:
df["days_to_trend"] = (df["trending_date"] - df["publish_time"].dt.normalize()).dt.days

print(df["days_to_trend"].describe().round(2))
print("\nShare of records that trended within 3 days:",
      round((df["days_to_trend"] <= 3).mean() * 100, 1), "%")
display(df["days_to_trend"].value_counts().sort_index().head(10))

count    46000.00
mean         5.09
std          6.24
min          0.00
25%          1.00
50%          3.00
75%          6.00
max         47.00
Name: days_to_trend, dtype: float64

Share of records that trended within 3 days: 55.3 %


,count
days_to_trend,
0,6136
1,7315
2,6625
3,5363
4,4122
5,3136
6,2384
7,1811
8,1390


In [36]:
video_summary = df.groupby(["country", "video_id"]).agg(
    title=("title", "first"),
    channel_title=("channel_title", "first"),
    category_name=("category_name", "first"),
    lifespan=("trending_date", "nunique"),
    max_views=("views", "max")
).reset_index()

print("Videos:", len(video_summary))
print(video_summary["lifespan"].describe().round(2))

Videos: 15059
count    15059.00
mean         3.05
std          2.92
min          1.00
25%          1.00
50%          2.00
75%          4.00
max         25.00
Name: lifespan, dtype: float64


In [37]:
top10_life = video_summary.nlargest(10, "lifespan")
display(top10_life[["title", "channel_title", "country", "category_name", "lifespan"]])

,title,channel_title,country,category_name,lifespan
950,Official Football Update,InsaneReaction46,GB,Entertainment,25
1019,KAIRO - FESTIVAL (LIVE),QuietChallenge75,GB,Music,25
1232,Kairo - Phones (Lyric Video)!,PerfectTrailer65,GB,Music,25
1432,Sunny Park - Cats (Live) MUST,FinalTrailer17,GB,Music,25
1611,Anya Lux - Paris (Lyric Video),QuietInterview9,GB,Music,25
1630,MIRA SOLIS - WINTER (LYRIC VIDEO)?,FinalTrailer17,GB,Music,25
1846,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,Music,25
2343,Nova Kane - Robots (Official Video),QuietChallenge75,GB,Music,25
2668,Final Cars Speech WATCH,UltimateUnboxing65,GB,Film & Animation,25
2687,Sunny Park - Robots (Live),SurprisingUpdate52,GB,Music,25


In [38]:
life_by_category = (video_summary.groupby("category_name")["lifespan"]
                    .agg(videos="count", avg_lifespan="mean")
                    .round(2)
                    .sort_values("avg_lifespan", ascending=False))
display(life_by_category)

,videos,avg_lifespan
category_name,,
Music,1933,5.33
Entertainment,3467,3.54
People & Blogs,1365,2.79
Comedy,1167,2.79
Film & Animation,850,2.70
Gaming,778,2.64
Education,674,2.44
Howto & Style,759,2.40
Autos & Vehicles,425,2.37


## 10. Combining data

In [39]:
clean_by_country = {c: df[df["country"] == c] for c in COUNTRIES}

for c, d in clean_by_country.items():
    print(c, d.shape)

stacked = pd.concat(clean_by_country.values(), ignore_index=True)

print("Stacked shape:", stacked.shape)
print("Same number of rows as df:", len(stacked) == len(df))

US (16000, 20)
GB (15000, 20)
IN (15000, 20)
Stacked shape: (46000, 20)
Same number of rows as df: True


In [40]:
channel_summary = video_summary.groupby("channel_title").agg(
    trending_videos=("video_id", "count"),
    total_views=("max_views", "sum")
).reset_index()

display(channel_summary.head())

,channel_title,trending_videos,total_views
0,AmazingBreakdown51,4,612225
1,AmazingChallenge30,3,358277
2,AmazingChallenge43,7,2344012
3,AmazingChallenge55,43,27824686
4,AmazingChallenge7,2,445985


In [41]:
best_video = (video_summary.sort_values("max_views", ascending=False)
              .drop_duplicates("channel_title")
              [["channel_title", "title", "max_views"]]
              .rename(columns={"title": "top_video", "max_views": "top_video_views"}))

display(best_video.head())

,channel_title,top_video,top_video_views
1846,FinalSpeech81,Lena Ortiz - Monsoon (Live),360760164
1787,WildUnboxing39,The Hollow Pines - Phones (Official Video) WATCH,343297002
2178,PerfectTrailer65,Rook & Vale - Phones (Official Video) WATCH,255362869
2722,CrazyChallenge57,Rook & Vale - Guitar (Live),168760418
11439,PerfectTutorial78,Lena Ortiz - Guitar (Official Video)!,129229245


In [42]:
channel_report = channel_summary.merge(best_video, on="channel_title", how="left")

print("Channels:", len(channel_report))
display(channel_report.sort_values("total_views", ascending=False).head(10))

Channels: 438


,channel_title,trending_videos,total_views,top_video,top_video_views
290,QuietChallenge75,473,1339792039,Kairo - Space (Lyric Video)!!,56884049
432,WildUnboxing39,57,582623682,The Hollow Pines - Phones (Official Video) WATCH,343297002
133,FinalTrailer17,166,504969156,Nova Kane - Football (Lyric Video),36232219
281,PerfectTutorial78,100,480943573,Lena Ortiz - Guitar (Official Video)!,129229245
278,PerfectTrailer65,67,453629190,Rook & Vale - Phones (Official Video) WATCH,255362869
132,FinalSpeech81,35,443010583,Lena Ortiz - Monsoon (Live),360760164
359,SurprisingHighlights41,128,403780627,Rook & Vale - Football (Lyric Video),119711498
248,OfficialReview41,468,398413392,Incredible Cats Secret,37902491
294,QuietInterview9,90,342485899,Anya Lux - Paris (Lyric Video),97513776
31,Brand NewChallenge45,310,311324020,Final Cricket Challenge,30277219


In [46]:
import os
os.makedirs("data_small", exist_ok=True)

for country in COUNTRIES:
    d = pd.read_csv(f"data/{country}videos.csv").drop(columns=["tags", "thumbnail_link"], errors="ignore")
    d.to_csv(f"data_small/{country}videos.csv", index=False)

print(os.listdir("data_small"))

['INvideos.csv', 'GBvideos.csv', 'USvideos.csv']


In [47]:
import shutil
shutil.make_archive("data_small", "zip", "data_small")

'/content/data_small.zip'